# NBA Churn Retention Engine — Exploratory Data Analysis

End-to-end walkthrough: data → features → model → SHAP → archetypes → NBA → impact.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')
print('Libraries loaded ✓')

## 1. Data Generation

In [ ]:
from src.data.generate_data import generate_dataset

df = generate_dataset(n=10000)
print(f'Shape: {df.shape}')
print(f'Churn rate: {df.churned.mean():.2%}')
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Churn rate by subscription type
churn_by_sub = df.groupby('subscription_type')['churned'].mean().sort_values(ascending=False)
churn_by_sub.plot(kind='bar', ax=axes[0], color='#e94560', alpha=0.85)
axes[0].set_title('Churn Rate by Subscription Type')
axes[0].set_ylabel('Churn Rate')
axes[0].tick_params(axis='x', rotation=30)

# MRR distribution
df['monthly_revenue'].hist(bins=50, ax=axes[1], color='#457b9d', alpha=0.8)
axes[1].set_title('Monthly Revenue Distribution')
axes[1].set_xlabel('MRR ($)')

# Tenure vs churn
churned = df[df.churned == 1]['tenure_months']
retained = df[df.churned == 0]['tenure_months']
axes[2].hist(retained, bins=30, alpha=0.6, label='Retained', color='#2ecc71')
axes[2].hist(churned, bins=30, alpha=0.6, label='Churned', color='#e74c3c')
axes[2].set_title('Tenure Distribution by Churn')
axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
from src.features.feature_engineering import engineer_features, ENGINEERED_FEATURES

df_eng = engineer_features(df)
print(f'Original features: {df.shape[1]}')
print(f'After engineering: {df_eng.shape[1]}')
print(f'New features: {ENGINEERED_FEATURES}')

In [ ]:
# Correlation of engineered features with churn
score_cols = ['engagement_decay_score', 'friction_score', 'pricing_sensitivity_score',
              'growth_pressure_score', 'customer_health_score', 'retention_risk_score']
corr = df_eng[score_cols + ['churned']].corr()['churned'].drop('churned').sort_values()

colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr.values]
corr.plot(kind='barh', color=colors)
plt.title('Engineered Feature Correlation with Churn')
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 3. Model Training & Comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from src.features.feature_engineering import MODEL_FEATURES
from src.models.train import train_all_models

X = df_eng[MODEL_FEATURES].fillna(0).values
y = df_eng['churned'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

best_model, metrics_df = train_all_models(
    X_train_s, y_train, X_test_s, y_test,
    feature_names=MODEL_FEATURES, tune_best=False
)
print('\nModel Comparison:')
print(metrics_df[['accuracy','precision','recall','f1','roc_auc','pr_auc']].to_string())

In [ ]:
# Plot ROC-AUC comparison
metrics_df['roc_auc'].sort_values().plot(
    kind='barh', color='#3498db', alpha=0.85, xlim=(0.7, 1.0)
)
plt.title('Model ROC-AUC Comparison')
plt.xlabel('ROC-AUC')
plt.axvline(0.9, color='red', linestyle='--', label='0.90 threshold')
plt.legend()
plt.tight_layout()
plt.show()

## 4. SHAP Explainability

In [ ]:
from src.explainability.shap_engine import SHAPExplainer

explainer = SHAPExplainer(best_model, feature_names=MODEL_FEATURES)
explainer.fit(X_train_s[:1000])

# Explain test set
sv = explainer.explain(X_test_s[:500])
importance_df = explainer.get_global_importance()
print('Top 10 features by SHAP importance:')
print(importance_df.head(10).to_string(index=False))

In [ ]:
import shap
shap.summary_plot(sv, X_test_s[:500], feature_names=MODEL_FEATURES, max_display=15)

## 5. Archetype Classification

In [ ]:
from src.nba_engine.archetype_classifier import classify_archetypes, ARCHETYPE_NAMES

churn_probs = best_model.predict_proba(X_test_s)[:, 1]
df_test = df_eng.iloc[len(X_train):].copy().reset_index(drop=True)
df_test['churn_probability'] = churn_probs

df_classified = classify_archetypes(df_test[df_test.churn_probability >= 0.5])
print(df_classified['churn_archetype'].value_counts())
print('\nArchetype distribution:')
print(df_classified.groupby('churn_archetype')['monthly_revenue'].mean().round(2))

## 6. NBA Recommendations & Business Impact

In [ ]:
from src.nba_engine.nba_engine import run_nba_pipeline
from src.clv.clv_calculator import add_clv_columns
from src.business_impact.impact_engine import simulate_business_impact, print_impact_report

df_with_clv = add_clv_columns(df_classified)
df_nba = run_nba_pipeline(df_with_clv)
report, df_impact = simulate_business_impact(df_nba)
print_impact_report(report)

In [ ]:
# Revenue saved by archetype
if not report.by_archetype.empty:
    report.by_archetype.set_index('churn_archetype')['revenue_saved'].plot(
        kind='bar', color=['#6c757d','#dc3545','#fd7e14','#198754'], alpha=0.85
    )
    plt.title('Expected Revenue Saved by Archetype')
    plt.ylabel('Revenue Saved ($)')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()